# 04 — Candidate generation (4 sources, cap 500/user)

Sources:
1. **History repeat** (~30): items user đã interact, weighted by recency × intent.
2. **Co-visitation** (~200, trục chính): cặp item cùng session trên tier A items.
3. **Popularity per segment** (~100): top-50 contacts_24h last 14d theo (category × city).
4. **Content-based cho cold** (~50): match (category, district, ad_type) cho tier C items.

Output: `candidates_valid.parquet` với cột `user_id, item_id, src_history, src_covis, src_pop, src_content`.

Target: `recall@500 ≥ 0.6` trên valid_users.

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time
import pandas as pd
import numpy as np

from utils.covis import build_covis, score_user_covis
from utils.candidates import (
    gen_history_candidates, gen_covis_candidates,
    gen_popularity_candidates, gen_content_candidates,
    merge_candidates,
)
from utils.metrics import mean_recall_at_k

t0 = time.time()
train_pos = pd.read_parquet(f"{CACHE_DIR}/train_pos.parquet")
train_pos["event_ts"] = pd.to_datetime(train_pos["event_ts"])
enr = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet")
valid_gt = pd.read_parquet(f"{CACHE_DIR}/valid_gt.parquet")
print(f"train_pos: {len(train_pos):,} | enr: {len(enr):,} | "
      f"valid_gt: {len(valid_gt):,} | {time.time()-t0:.1f}s")

VALID_USERS = set(valid_gt["user_id"].unique())
print(f"|valid users|: {len(VALID_USERS):,}")

In [ ]:
# Limit train_pos to valid users to save memory in candidate gen
train_pos_valid = train_pos[train_pos["user_id"].isin(VALID_USERS)].copy()
print(f"train_pos for valid users: {len(train_pos_valid):,}")

allowed_a = set(enr[enr["tier"] == "A"]["item_id"])
allowed_ab = set(enr[enr["tier"].isin(["A", "B"])]["item_id"])
print(f"|tier A|: {len(allowed_a):,} | |tier A+B|: {len(allowed_ab):,}")

In [ ]:
# ---- Source 1: history (tier A+B allowed) ----
t0 = time.time()
hist_cands = gen_history_candidates(train_pos_valid, allowed_ab, top_n=30)
print(f"history: {sum(len(v) for v in hist_cands.values()):,} pairs | {time.time()-t0:.1f}s")

In [ ]:
# ---- Source 2: co-visitation (tier A only) ----
# Build covis from train_pos (all users, not just valid - more signal)
t0 = time.time()
covis_input = train_pos[train_pos["item_id"].isin(allowed_a)][
    ["user_id", "session_id", "item_id", "event_type", "event_ts"]
]
print(f"covis input rows: {len(covis_input):,}")
covis = build_covis(covis_input, allowed_items=allowed_a,
                    top_k_per_item=20, time_decay=True)
print(f"covis items: {len(covis):,} | {time.time()-t0:.1f}s")

t0 = time.time()
covis_cands = gen_covis_candidates(train_pos_valid, covis,
                                    allowed_items=allowed_a, top_n=200)
print(f"covis cands: {sum(len(v) for v in covis_cands.values()):,} pairs | {time.time()-t0:.1f}s")

In [ ]:
# ---- Source 3: popularity per segment ----
# Build user_profile from train_pos: top category/city/district/ad_type per user
t0 = time.time()
enr_seg_cols = [c for c in ["item_id", "category", "city_name",
                             "district_name", "ad_type"] if c in enr.columns]
tp = train_pos_valid.merge(enr[enr_seg_cols], on="item_id", how="left")

def _mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else None

prof_specs = {}
for src, out in [("category", "u_top_category"), ("city_name", "u_top_city"),
                  ("district_name", "u_top_district"), ("ad_type", "u_top_ad_type")]:
    if src in tp.columns:
        prof_specs[out] = (src, _mode)
user_profile = tp.groupby("user_id").agg(**prof_specs).reset_index()
print(f"user_profile: {len(user_profile):,} | {time.time()-t0:.1f}s")

snap = pd.read_parquet(f"{CACHE_DIR}/snapshot_60d.parquet")
snap["date"] = pd.to_datetime(snap["date"])
last14_start = pd.Timestamp(TRAIN_DATE_END) - pd.Timedelta(days=14)
snap_14 = snap[snap["date"] >= last14_start]
print(f"snap_14d: {len(snap_14):,}")

pop_cands = gen_popularity_candidates(user_profile, enr, snap_14,
                                       top_n_per_seg=50, top_n_per_user=100)
print(f"pop cands: {sum(len(v) for v in pop_cands.values()):,} pairs")

In [ ]:
# ---- Source 4: content-based for cold items ----
t0 = time.time()
content_cands = gen_content_candidates(user_profile, enr, top_n=50)
print(f"content cands: {sum(len(v) for v in content_cands.values()):,} pairs | {time.time()-t0:.1f}s")

In [ ]:
# ---- Merge + cap 500 / user ----
t0 = time.time()
cands = merge_candidates({
    "history": hist_cands,
    "covis": covis_cands,
    "pop": pop_cands,
    "content": content_cands,
}, cap_total=500)
print(f"candidates: {len(cands):,} | users: {cands['user_id'].nunique():,} | "
      f"avg cand/user: {len(cands)/max(cands['user_id'].nunique(),1):.1f} | "
      f"{time.time()-t0:.1f}s")

# ---- Filter out (user, item) pairs where user already 'purchased' the listing ----
_pci = pd.read_parquet(f"{CACHE_DIR}/pci_full.parquet", columns=["user_id", "item_id", "purchased"])
_purchased = _pci[_pci["purchased"] == True][["user_id", "item_id"]].drop_duplicates()
_purchased["_drop"] = True
_before = len(cands)
cands = cands.merge(_purchased, on=["user_id", "item_id"], how="left")
cands = cands[cands["_drop"].isna()].drop(columns=["_drop"])
print(f"Purchased filter: {_before - len(cands):,} rows removed -> {len(cands):,} remain")
del _pci, _purchased

cands.to_parquet(f"{CACHE_DIR}/candidates_valid.parquet", index=False)

# Verify no tier-D leakage
chk = cands.merge(enr[["item_id", "tier"]], on="item_id", how="left")
assert chk["tier"].isin(["A", "B", "C"]).all(), \
    f"Tier D leakage: {chk[chk['tier']=='D'].shape[0]}"
print("No tier-D leakage. OK.")

In [ ]:
# ---- Compute recall@500 on valid ----
gt = (valid_gt.groupby("user_id")["item_id"].agg(set).to_dict())
cand_by_user = (cands.groupby("user_id")["item_id"].apply(list).to_dict())
r500 = mean_recall_at_k(cand_by_user, gt, k=500)
print(f"Recall@500 (ceiling for ranker): {r500:.4f}")
if r500 < 0.5:
    print("WARNING: recall@500 below 0.5 — ranker can't recover. "
          "Consider widening candidate sources.")